# Market Basket Analysis (Algoritma Apriori) - All-in-One: EDA, Modeling & Streamlit App

## Tujuan Pembelajaran & Konteks Proyek
Notebook ini mempelajari **Unsupervised Learning - Association Rule Mining (Market Basket Analysis)** untuk menemukan pola keterkaitan antar barang yang dibeli secara bersamaan oleh pelanggan toko roti (`bread_basket.csv`).

### Mengapa Menggunakan Algoritma *Apriori*?
1. **Menemukan Pola *"Jika Membeli A, Maka Membeli B"*:** Berbeda dengan regresi (memprediksi angka) atau klastering (mengelompokkan pelanggan), *Market Basket Analysis* mencari aturan asosiasi (*Association Rules*) antar produk di dalam keranjang belanja (`Transaction`).
2. **Prinsip Apriori (*Downward Closure Property*):** Jika suatu kombinasi barang jarang dibeli (nilai *support*-nya di bawah ambang batas `min_support`), maka kombinasi yang lebih besar yang memuat barang tersebut pasti juga jarang dibeli. Prinsip ini memangkas jutaan kombinasi kandidat sehingga komputasi jauh lebih efisien.
3. **Tiga Metrik Utama yang Wajib Dipahami:**
   - **Support ($P(A \cap B)$):** Seberapa sering item $A$ dan $B$ muncul bersama dari seluruh transaksi. Pada proyek ini digunakan `min_support = 0.01` ($1\%$ dari seluruh transaksi).
   - **Confidence ($P(B|A) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$):** Seberapa besar peluang pelanggan membeli item $B$ **jika** mereka sudah membeli item $A$.
   - **Lift ($\frac{\text{Confidence}(A \rightarrow B)}{\text{Support}(B)}$):** Mengukur kekuatan hubungan sebenarnya dibanding kebetulan semata:
     - $\text{Lift} > 1$: Item $A$ dan $B$ saling mendukung (pembelian $A$ meningkatkan peluang pembelian $B$).
     - $\text{Lift} = 1$: Keduanya independen (tidak berhubungan).
     - $\text{Lift} < 1$: Keduanya saling mensubstitusi (jarang dibeli bersamaan).

### Trade-offs (Kelebihan vs Keterbatasan)
- **Kelebihan:** Hasil aturan sangat mudah dipahami oleh pelaku bisnis untuk penataan rak toko (*product placement*), paket *bundling*, dan sistem rekomendasi silang (*cross-selling*).
- **Keterbatasan:** Membutuhkan matriks transaksi biner (*One-Hot Pivot Table* 0/1) dan bisa menjadi lambat jika jumlah item unik mencapai puluhan ribu dengan `min_support` terlalu kecil (alternatif untuk skala sangat besar: *FP-Growth*).

**Persiapan Awal:**
Jika file **`bread_basket.csv`** belum ada di folder kerja, notebook ini akan mengunduhnya secara otomatis dari Kaggle (`mittalvasu95/the-bread-basket`).

--- 
# BAGIAN 1: Exploratory Data Analysis & Pemodelan Apriori (Part 1)
## 1.1 Install, Import Library & Load Dataset

In [ ]:
%pip install -q numpy pandas matplotlib seaborn mlxtend streamlit kagglehub

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Auto-download bread_basket.csv dari Kaggle jika belum tersedia di folder lokal
if not os.path.exists('bread_basket.csv'):
    import kagglehub
    dl_path = kagglehub.dataset_download('mittalvasu95/the-bread-basket')
    for fname in os.listdir(dl_path):
        if fname.endswith('.csv'):
            shutil.copy(os.path.join(dl_path, fname), 'bread_basket.csv')
            break

# Membaca dataset bread_basket.csv
data = pd.read_csv('bread_basket.csv')
print('DataFrame shape :', data.shape)
data.head()

## 1.2 Konversi & Ekstraksi Format Data Waktu (`date_time`)

In [ ]:
# Format data waktu
data['date_time'] = pd.to_datetime(data['date_time'], format='%d-%m-%Y %H:%M')
print('Tipe data date_time:', data['date_time'].dtype)

# Ekstraksi fitur bulan (month), hari (day), dan jam (hour)
data['month'] = data['date_time'].dt.month
data['day'] = data['date_time'].dt.weekday
data['hour'] = data['date_time'].dt.hour
data.head()

## 1.3 Visualisasi Data (EDA)

In [ ]:
# 1. Menampilkan 10 item paling laris
plt.figure(figsize=(13, 5))
sns.set_palette('muted')

sns.barplot(
    x=data['Item'].value_counts()[:10].index,
    y=data['Item'].value_counts()[:10].values
)
plt.xlabel('')
plt.ylabel('')
plt.xticks(size=13, rotation=45)
plt.title('10 Produk yang Paling Laris', size=17)
plt.show()

In [ ]:
# 2. Transaksi tiap bulan (Oktober - April)
data_perbulan = data.groupby('month')['Transaction'].count()
data_perbulan = pd.concat([data_perbulan.iloc[4:], data_perbulan.iloc[:4]])

plt.figure(figsize=(8, 5))
sns.barplot(
    x=['October', 'November', 'December', 'January', 'February', 'March', 'April'],
    y=data_perbulan.values,
    color='#D5AAD3'
)
plt.xticks(size=12, rotation=-30)
plt.title('Jumlah Transaksi Tiap Bulan dari Oktober - April', size=16)
plt.show()

In [ ]:
# 3. Transaksi per hari (Monday - Sunday)
data_perday = data.groupby('day')['Transaction'].count()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
    y=data_perday.values,
    color='#BFFCC6'
)
plt.xticks(size=12, rotation=-30)
plt.title('Total Transaksi Per Hari', size=16)
plt.show()

In [ ]:
# 4. Pembelian tiap jam
data_perhour = data.groupby('hour')['Transaction'].count()

plt.figure(figsize=(12, 6))
sns.barplot(
    x=data_perhour.index,
    y=data_perhour.values,
    color='#85E3FF'
)
plt.xlabel('Hour', size=15)
plt.title('Total Transaksi Per Jam', size=17)
plt.xticks(size=13)
plt.show()

## 1.4 Data Preparation & Pemodelan Algoritma Apriori

In [ ]:
# Membersihkan teks pada kolom Item (lowercase & strip whitespace)
data['Item'] = data['Item'].apply(lambda item: item.lower())
data['Item'] = data['Item'].apply(lambda item: item.strip())

# Mengambil kolom Transaction dan Item saja
data = data[['Transaction', 'Item']].copy()
data.head(10)

In [ ]:
from mlxtend.frequent_patterns import association_rules, apriori

# Menghitung jumlah item per transaksi
item_count = data.groupby(['Transaction', 'Item'])['Item'].count().reset_index(name='Count')

# Membuat Pivot Table (Transaction vs Item)
item_count_pivot = item_count.pivot_table(index='Transaction', columns='Item', values='Count', aggfunc='sum').fillna(0)
item_count_pivot = item_count_pivot.astype('int32')

# Fungsi encoding biner (0 = tidak dibeli, 1 = dibeli)
def encode(x):
    if x <= 0:
        return 0
    elif x >= 1:
        return 1

item_count_pivot = item_count_pivot.map(encode) if hasattr(item_count_pivot, 'map') else item_count_pivot.applymap(encode)
print('Ukuran dataset :', item_count_pivot.shape)
print('Jumlah transaksi :', item_count_pivot.shape[0])
print('Jumlah item :', item_count_pivot.shape[1])
item_count_pivot.head()

In [ ]:
# Menjalankan Algoritma Apriori dengan minimum support 0.01 (1%)
support = 0.01
frequent_items = apriori(item_count_pivot.astype(bool), min_support=support, use_colnames=True)
frequent_items.sort_values('support', ascending=False).head(10)

In [ ]:
# Membuat Association Rules berdasarkan metrik Lift
metric = 'lift'
min_threshold = 1

rules = association_rules(frequent_items, metric=metric, min_threshold=min_threshold)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules.sort_values('confidence', ascending=False, inplace=True)
rules.head(15)

---
# BAGIAN 2: Pembuatan & Eksekusi Aplikasi Web Streamlit (Part 2)
Bagian di bawah ini menuliskan file `apriori.py` secara otomatis lalu menjalankannya di `http://localhost:8501` (VS Code Lokal) atau via Cloudflare Tunnel (Google Colab).

## 2.1 Buat File `apriori.py` Secara Otomatis

In [ ]:
%%writefile apriori.py
import streamlit as st
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import association_rules, apriori

# Load dataset
df = pd.read_csv('bread_basket.csv')
df['date_time'] = pd.to_datetime(df['date_time'], format='%d-%m-%Y %H:%M')

df['month'] = df['date_time'].dt.month
df['day'] = df['date_time'].dt.weekday

df['month'] = df['month'].replace([i for i in range(1, 12 + 1)], [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
])

df['day'] = df['day'].replace([i for i in range(6 + 1)], [
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

st.title('Market Basket Analysis Menggunakan Algoritma Apriori')

def get_data(period_day='', weekday_weekend='', month='', day=''):
    data = df.copy()
    filtered = data.loc[
        (data['period_day'].str.contains(period_day)) &
        (data['weekday_weekend'].str.contains(weekday_weekend)) &
        (data['month'].str.contains(month.title())) &
        (data['day'].str.contains(day.title()))
    ]
    return filtered if filtered.shape[0] else 'No Result!'

def user_input_features():
    item = st.selectbox('Item', [
        'Bread', 'Scandinavian', 'Hot chocolate', 'Jam', 'Cookies', 'Muffin',
        'Coffee', 'Pastry', 'Medialuna', 'Tea', 'Tartine', 'Basket',
        'Mineral water', 'Farm House', 'Fudge', 'Juice', "Ella's Kitchen Pouches",
        'Victorian Sponge', 'Frittata', 'Hearty & Seasonal', 'Soup',
        'Pick and Mix Bowls', 'Smoothies', 'Cake', 'Mighty Protein', 'Chicken sand',
        'Coke', 'My-5 Fruit Shoot', 'Focaccia', 'Sandwich', 'Alfajores', 'Eggs',
        'Brownie', 'Dulce de Leche', 'Honey', 'The BART', 'Granola',
        'Fairy Doors', 'Empanadas', 'Keeping It Local', 'Art Tray', 'Bowl Nic Pitt',
        'Bread Pudding', 'Adjustment', 'Truffles', 'Chimichurri Oil', 'Bacon',
        'Spread', 'Kids biscuit', 'Siblings', 'Caramel bites', 'Jammie Dodgers',
        'Tiffin', 'Olum & polenta', 'Polenta', 'The Nomad', 'Hack the stack',
        'Bakewell', 'Lemon and coconut', 'Toast', 'Scone', 'Crepes', 'Vegan mincepie',
        'Bare Popcorn', 'Muesli', 'Crisps', 'Pintxos', 'Gingerbread syrup', 'Panatone',
        'Brioche and salami', 'Afternoon with the baker', 'Salad', 'Chicken Stew',
        'Spanish Brunch', 'Raspberry shortbread sandwich', 'Extra Salami or Feta',
        'Duck egg', 'Baguette', "Valentine's card", 'Tshirt', 'Vegan Feast',
        'Postcard', 'Nomad bag', 'Chocolates', 'Coffee granules',
        'Drinking chocolate spoons', 'Christmas common', 'Argentina Night',
        'Half slice Monster', 'Gift voucher', 'Cherry me Dried fruit', 'Mortimer',
        'Raw bars', 'Tacos/Fajita'
    ])
    period_day = st.selectbox('Period Day', ['Morning', 'Afternoon', 'Evening', 'Night'])
    weekday_weekend = st.selectbox('Weekday / Weekend', ['Weekend', 'Weekday'])
    month = st.select_slider('Month', [
        'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
    ])
    day = st.select_slider('Day', [
        'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'
    ], value='Sat')

    return period_day, weekday_weekend, month, day, item

period_day, weekday_weekend, month, day, item = user_input_features()

data = get_data(period_day.lower(), weekday_weekend.lower(), month, day)

def encode(x):
    if x <= 0:
        return 0
    elif x >= 1:
        return 1

if type(data) != type('No Result!'):
    item_count = data.groupby(['Transaction', 'Item'])['Item'].count().reset_index(name='Count')
    item_count_pivot = item_count.pivot_table(index='Transaction', columns='Item', values='Count', aggfunc='sum').fillna(0)
    item_count_pivot = item_count_pivot.map(encode) if hasattr(item_count_pivot, 'map') else item_count_pivot.applymap(encode)

    support = 0.01
    frequent_items = apriori(item_count_pivot.astype(bool), min_support=support, use_colnames=True)

    metric = 'lift'
    min_threshold = 1

    rules = association_rules(frequent_items, metric=metric, min_threshold=min_threshold)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    rules.sort_values('confidence', ascending=False, inplace=True)

def parse_list(x):
    x = list(x)
    if len(x) == 1:
        return x[0]
    elif len(x) > 1:
        return ', '.join(x)

def return_item_df(item_antecedents):
    data = rules[['antecedents', 'consequents']].copy()
    data['antecedents'] = data['antecedents'].apply(parse_list)
    data['consequents'] = data['consequents'].apply(parse_list)

    matching_rows = data.loc[data['antecedents'] == item_antecedents]
    if not matching_rows.empty:
        return list(matching_rows.iloc[0, :])
    return []

if type(data) != type('No Result!'):
    st.markdown('Hasil Rekomendasi :')
    result = return_item_df(item)
    if result:
        st.success(f"Jika Konsumen Membeli **{item}**, maka membeli **{result[1]}** secara bersamaan")
    else:
        st.warning(f"Tidak ditemukan aturan asosiasi untuk item **{item}** pada filter waktu yang dipilih.")
else:
    st.warning('Tidak ada data transaksi yang cocok dengan filter waktu tersebut (No Result!).')

## 2.2 Jalankan Server Streamlit (VS Code Lokal & Google Colab)
Sel di bawah ini otomatis menyesuaikan environment: membuka di `http://localhost:8501` jika dijalankan di VS Code lokal, atau membuat tautan `.trycloudflare.com` jika dijalankan di Google Colab.

In [ ]:
import sys
import subprocess
import time

IN_COLAB = 'google.colab' in sys.modules

streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "apriori.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

if IN_COLAB:
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
    print('Membuka Cloudflare Tunnel... Klik link .trycloudflare.com yang muncul di bawah ini:')
    !./cloudflared tunnel --url http://localhost:8501 --no-autoupdate
else:
    print('Server Streamlit telah berjalan di background (VS Code Lokal)!')
    print('Buka di browser: http://localhost:8501')